<div
  style="
    background-color: #f0f0f0;
    color:rgb(56, 56, 56);
    padding: 8px;
    display: flex;
    align-items: center;
    gap: 100px;
  "
>
  <img src="./images/brand.svg" style="max-height: 80px;">
  <strong>
    AI Saga: Deep Learning & Generative AI</br>
    1.Lab Building a Multi-Layer Perceptron from Scratch for Wine Quality Classification</br>
  </strong>
  <emph>
    Student Name: [Complete with your name]</br>
    Date: [Fill up with the submission date]</br>
  </emph>
</div>

### Background

You will be working with the **Wine Quality Dataset**, which contains physicochemical measurements of Portuguese "Vinho Verde" wines.

Each wine sample is rated by experts on a quality scale from 3 to 9. Your task is to build a custom multi-layer perceptron (MLP) using PyTorch to classify wines into **three quality categories**:

- **Low Quality**: scores 3-5 (below average wines)
- **Medium Quality**: scores 6-7 (average wines)
- **High Quality**: scores 8-9 (premium wines)

---

### Description

Check LMS for the last updated instructions.

Don't forget to **convert** the actual dataset into a 3-class one.

---

### Deliverables

Complete the Jupyter Notebook file named **1.lab.1-wine-quality-mlp.ipynb**.

#### Requirements

- Use **PyTorch** high-level libraries
- Code must be well-commented and clearly organized
- Include markdown cells explaining each step and your design decisions
- You can use Google Colab and their free-tier GPU

---

### Key Reminders

**Think before implementing your MLP class** - consider the input/output dimensions, activation functions, and architectural choices carefully

**Present well-reasoned conclusions** - your analysis section should demonstrate deep understanding of your model's behavior and performance. **This last section is critical for demonstrating deep understanding.** Discuss:

- Which features appear most important for wine quality classification?
- How well does your model distinguish between the three quality classes?
- What challenges did you encounter?
- What architectural choices or hyperparameters had the most impact on performance?
- Suggestions for improvement

**Note**: All submitted code is code that you must be able to defend (even if AI generated!).

In [ ]:
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

import matplotlib.pyplot as plt
import seaborn as sns

from rich.table import Table
from rich.console import Console

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

### Initial configuration

Visual configuration of Seaborn for more attractive graphics

In [ ]:
sns.set_theme(style="whitegrid", palette="muted")
sns.set_context("notebook", font_scale=1.2)

Rich console instance to display results

In [ ]:
console = Console()

If the system has a CUDA-compatible GPU, the GPU is selected, otherwise, the CPU is selected.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
console.print(f"Using device: {device}")

### Reading Dataset

Lets read the dataset using pandas.

In [ ]:
df = pd.read_csv("./data/wine-quality-dataset.csv")
df.head()

### Analyzing the Data

1. **df.shape():** This function help us to understand the number of rows (observations) and columns (features) in the dataset. This gives an overview of the dataset's size and structure.

In [ ]:
console.print(df.shape)

2. **df.info():** This function help us to understand the dataset by showing the number of records in each column, type of data, whether any values are missing and how much memory the dataset uses.

In [ ]:
df.info()

3. **df.nunique():** This function tells us how many unique values exist in each column which provides insight into the variety of data in each feature.

In [ ]:
df.nunique()

4. **df.describe().T**: This method give me a statistical summary of the DataFrame (Transpose) showing values like count, mean, standard deviation, minimum and quartiles for each numerical column. It helps in summarizing the central tendency and spread of the data.

In [ ]:
df.describe().T

5. **df.columns.tolist():** This converts the column names of the DataFrame into a Python list making it easy to access and manipulate the column names.

In [ ]:
df.columns.tolist()

6. **df.isnull().sum():** This function help us to check the number of missing (null) values in each column of the dataset.

In [ ]:
df.isnull().sum()

7. **df.isna().sum():** This function help us for the same purpose as .isnull(), since both are equivalent and return the count of NaN or undefined values per column.

In [ ]:
df.isna().sum()

8. **sns.boxplot():** This function help us to visualize the distribution of numerical data and identify possible outliers.

In [ ]:
plt.figure(figsize=(16, 6))
sns.boxplot(data=df)
plt.xticks(rotation=90)
plt.show()

### Observations

So far, the dataset has been explored to understand its structure and data quality. It contains 1,143 records and 13 features. All columns have values of type float, except for quality and id. Additionally, there are no missing or NaN values in the dataset.

# 1. Exploratory Data Analysis (EDA)

We will follow these phases:

1. **Output variable analysis:** distribution of the quality.
2. **Univariate analysis:** distribution of individual features.
3. **Bivariate analysis:** relationship between features and the output variable.
4. **Correlation matrix and heatmap:** to identify multicollinearity and strong associations.
5. **Joint exploration with PairPlot:** to visualize patterns and separability across multiple features simultaneously.

## 1. Output Variable Analysis

**Explanation:**

- **sns.histplot(...):** Creates a histogram of the quality variable. 
- **sns.boxplot(..):** Generates a boxplot of the quality variable. This graph is useful for identifying the median, quartiles, and possible outliers.
- **plt.title(...) and plt.xlabel(...) / plt.ylabel(...):** These commands add descriptive titles and labels to the axes to improve the readability and understanding of each graph.

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
sns.histplot(df["quality"], bins=10, kde=True, color="teal")
plt.title("Quality Distribution (Histogram)", fontsize=14)
plt.xlabel("Quality")
plt.ylabel("Frequency")
plt.grid(axis="y", linestyle="--", alpha=0.6)

plt.subplot(1, 2, 2)
sns.boxplot(y=df["quality"], color="teal")
plt.title("Boxplot of Quality")
plt.ylabel("Quality")

plt.tight_layout()
plt.show()

## Observations 

The target variable quality ranges from 3 to 8, with most samples concentrated between 5 and 7. This shows a class imbalance, as mid-quality wines dominate the dataset. According to the boxplot, there are a few outliers at both ends, particularly for wines rated 3 and 8, which represent extreme quality cases.
This imbalance and the presence of outliers should be taken into account when training the neural network, since they may influence how the model learns and generalizes.

## 2. Univariate Analysis: Distribution of Characteristics

**Explanation:**

  * **plt.subplot(n_rows, n_cols, i):** Divides the figure into a grid of n rows and n columns, assigning each graph to a position.
  * **sns.histplot(df[characteristic], kde=True, bins=30):** Plots the histogram of each characteristic with a density curve to visualize the distribution.
  * **plt.tight_layout():** Optimizes the spacing between subplots to avoid overlap.

In [ ]:
n_cols = 4
n_rows = 4

plt.figure(figsize=(20, 20))

for i, feature in enumerate(df.columns, 1):
    plt.subplot(n_rows, n_cols, i)
    sns.histplot(df[feature], kde=True, bins=30)
    plt.title(f"Distribution of {feature}")
    plt.xlabel(feature)
    plt.ylabel("Frequency")

plt.tight_layout()
plt.show()

## Observations

The univariate analysis shows that the features have different distribution patterns:

* **Fixed acidity:** Slightly **left-skewed**, indicating that most wines have moderate to high acidity levels.
* **Volatile acidity:** Shows a **more symmetric** distribution, meaning most values are evenly spread around the center.
* **Citric acid:** **Right-skewed**, with most wines having lower citric acid levels and few high-value outliers.
* **Residual sugar:** **Right-skewed**, suggesting that most wines are dry, while a few contain higher sugar content.
* **Chlorides:** Displays a **mostly symmetric** distribution, but with **outliers** creating a long right tail.
* **Free sulfur dioxide:** **Right-skewed**, indicating that lower concentrations are more common.
* **Total sulfur dioxide:** Also **right-skewed**, with a few samples showing very high concentrations.
* **Density:** Roughly **symmetric**, centered around typical wine density values.
* **pH:** **Symmetric** distribution, showing that pH values are well-balanced across samples.
* **Sulphates:** **Right-skewed**, meaning most wines contain low to moderate sulphate levels.
* **Alcohol:** **Right-skewed**, showing that lower-alcohol wines are more frequent, with fewer high-alcohol samples.

Overall, several chemical features (like citric acid, residual sugar, sulfur compounds, sulphates, and alcohol) show positive skewness, suggesting the presence of a few extreme values. This pattern indicates the potential need for scaling or normalization before training the neural network to ensure all features contribute proportionally.

## 3. Bivariate Analysis: Relationship between Features and Quality

**Explanation**

In this phase, we analyze how each chemical property of the wine relates to the output variable `quality`.

* `sns.scatterplot(...)`:

  * `x=df[feature]`: Defines the variable on the X-axis.
  * `y=df['quality']`: Defines the target variable on the Y-axis.
* These plots help visualize whether there is a relationship (for example, linear or non-linear) between the wine’s chemical attributes and its quality score.

We will ignore the **`id`** column, since it does not provide useful information for prediction.

In [ ]:
n_cols = 4
n_rows = 3

# Exclude irrelevant columns
subset = [col for col in df.columns if col not in ["Id", "quality"]]

plt.figure(figsize=(20, 15))

for i, feature in enumerate(subset, 1):
    plt.subplot(n_rows, n_cols, i)
    sns.scatterplot(x=df[feature], y=df["quality"], alpha=0.6)
    plt.title(f"Quality vs {feature}")
    plt.xlabel(feature)
    plt.ylabel("Quality")

plt.tight_layout()
plt.show()

## Observations

Since the target variable quality only takes discrete values between 3 and 8, the scatter plots do not clearly show visible trends or patterns between the features and wine quality. Because of this, it is difficult to visually assess linear or non-linear relationships using this type of plot.

To obtain a clearer understanding of how each variable relates to wine quality, the analysis will continue with the next phase.

## 4. Correlation Matrix and Heatmap

**Explanation**

* **`df.corr()`**: Computes the **Pearson correlation coefficient** between all numerical variables.
* **`sns.heatmap(...)`**: Displays the correlation matrix as a heatmap for easier interpretation.

  * **`cmap='coolwarm'`**: Uses a color palette that highlights positive and negative correlations.
  * **`annot=True`**: Shows the correlation values within each cell.
  * **`fmt=".2f"`**: Formats the numbers to two decimal places.
  * **`linewidths=0.5`**: Adds separation lines between cells for better readability.

In [ ]:
matrix_correlation = df.drop(columns=["Id"]).corr()

plt.figure(figsize=(10, 8))
sns.heatmap(matrix_correlation, cmap="coolwarm", annot=True, fmt=".2f", linewidths=0.5)
plt.title("Correlation Matrix between Wine Features", fontsize=14)
plt.show()

### **Observations**

* **Quality** shows a **strong positive correlation with Alcohol** and a negative correlation with Volatile Acidity, confirming the expected trends for higher-quality wines.
* **Density** is **negatively correlated with Alcohol**, as wines with higher alcohol content tend to have lower density.
* **Free sulfur dioxide** and **Total sulfur dioxide** present a **moderate positive correlation**, indicating a consistent relationship between these two features.
* **Citric acid** has a **mild positive correlation** with both **Fixed acidity** and **Quality**.
* **Fixed acidity** shows a **positive correlation** with **Citric acid** and **Density**, and a **negative correlation** with **pH**, which is consistent with chemical behavior since higher acidity lowers the pH value.
* Most other variables exhibit **weak or near-zero correlations** with `quality`, suggesting a limited direct influence on the target variable.

Overall, this analysis confirms that **Alcohol** and **Volatile Acidity** are among the most relevant predictors of wine quality

## 5. Joint Exploration: PairPlot

**Explanation**

* **`sns.pairplot()`**: Creates a grid of plots that shows each variable against all others.

  * **`diag_kind='kde'`**: Displays Kernel Density Estimation (KDE) plots on the diagonal instead of histograms, which helps better visualize each variable’s distribution.
* This visualization is particularly useful for identifying **clusters**, **linear relationships**, or **non-linear trends** across multiple variables simultaneously.

In [ ]:
# Select key features based on correlation results
subset = [
    "fixed acidity",
    "volatile acidity",
    "citric acid",
    "residual sugar",
    "density",
    "pH",
    "alcohol",
    "quality",
]

sns.pairplot(df[subset], diag_kind="kde", hue="quality", palette="viridis")
plt.suptitle("PairPlot of Selected Wine Features", y=1.02)
plt.show()

## Observations

The diagonal KDE plots show that many variables do not follow a normal distribution, supporting the need for scaling or normalization prior to neural network training. In general, no perfectly separable clusters are observed, suggesting that the relationships are complex and nonlinear.

# 2. Preprocessing of data

Split features (X) and target (y)

In [ ]:
def convert_to_3_classes(quality):
    """
    - Low (0): 3-5
    - Medium (1): 6-7
    - High (2): 8-9
    """
    if quality <= 5:
        return 0
    elif quality <= 7:
        return 1
    else:
        return 2


X = df.drop(columns=["Id", "quality"])
y = df["quality"].apply(convert_to_3_classes)

Split into train (70%), validation (15%), and test (15%)

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

console.print(f"Train: {len(X_train)} \nValidation: {len(X_val)} \nTest: {len(X_test)}")

Method to escalate the data

In [ ]:
def scale_data(scaler, X_train, X_val, X_test):
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_val_scaled, X_test_scaled

Method to convert to tensors

In [ ]:
def to_tensors(X_train, y_train, X_val, y_val, X_test, y_test):
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32)

    y_train_tensor = torch.tensor(y_train.values, dtype=torch.long)
    y_val_tensor = torch.tensor(y_val.values, dtype=torch.long)
    y_test_tensor = torch.tensor(y_test.values, dtype=torch.long)

    return (
        X_train_tensor,
        y_train_tensor,
        X_val_tensor,
        y_val_tensor,
        X_test_tensor,
        y_test_tensor,
    )

Method to train and convert to tensors

In [ ]:
def preprocess_data(scaler, X_train, X_val, X_test, y_train, y_val, y_test):
    X_train_scaled, X_val_scaled, X_test_scaled = scale_data(
        scaler, X_train, X_val, X_test
    )
    return to_tensors(
        X_train_scaled, y_train, X_val_scaled, y_val, X_test_scaled, y_test
    )

In [ ]:
scaler = StandardScaler()
(
    X_train_tensor,
    y_train_tensor,
    X_val_tensor,
    y_val_tensor,
    X_test_tensor,
    y_test_tensor,
) = preprocess_data(scaler, X_train, X_val, X_test, y_train, y_val, y_test)

# 3. Implementation of Custom MLP

### Configurations

We will be explored multiple neural network architectures to determine which one provides the best performance. Because the optimal configuration is uncertain, different combinations of layers and neurons will be tested, and the model showing the best metric results will be selected.

In [ ]:
architectures = {
    "3-Layers (64-32-16)": [64, 32, 16],
    "3-Layers (128-64-32)": [128, 64, 32],
    "4-Layers (128-64-32-16)": [128, 64, 32, 16],
    "4-Layers (256-128-64-32)": [256, 128, 64, 32],
}

### Model Architectures

In [ ]:
class WineQualityNN(nn.Module):
    def __init__(self, input_size, hidden_sizes, output_size):
        super(WineQualityNN, self).__init__()

        layers = []
        last_size = input_size

        for i, h in enumerate(hidden_sizes):
            layers.append(nn.Linear(last_size, h))
            layers.append(nn.ReLU())
            last_size = h

        layers.append(nn.Linear(last_size, output_size))

        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

# 4. Training Loop

Training method

In [ ]:
def train_nn_model(model, X_train, y_train, X_val, y_val, epochs=100, lr=0.001):
    train_loss_history = []
    val_loss_history = []
    val_acc_history = []

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()

        y_train_logits = model(X_train)
        loss = criterion(y_train_logits, y_train)
        loss.backward()
        optimizer.step()

        train_loss_history.append(loss.item())

        model.eval()
        with torch.no_grad():
            y_val_logits = model(X_val)
            val_loss = criterion(y_val_logits, y_val)
            y_val_pred = torch.argmax(y_val_logits, dim=1)
            val_acc = accuracy_score(y_val.cpu().numpy(), y_val_pred.cpu().numpy())

        val_loss_history.append(val_loss.item())
        val_acc_history.append(val_acc)

    return model, train_loss_history, val_loss_history, val_acc_history

Method to evaluate the neural network

In [ ]:
def evaluate_nn_model(model, X_tensor):
    model.eval()
    with torch.no_grad():
        y_logits = model(X_tensor)
        y_pred = torch.argmax(y_logits, dim=1).cpu().numpy()
    return y_pred

Method to calculate metrics

In [ ]:
def calculate_metrics(y_true, y_pred):

    return {
        "accuracy_score": accuracy_score(y_true, y_pred),
        "precision_score": precision_score(
            y_true, y_pred, average="weighted", zero_division=0
        ),
        "recall_score": recall_score(
            y_true, y_pred, average="weighted", zero_division=0
        ),
        "f1_score": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }

Method to create the comparison table

In [ ]:
def create_comparison_table(title, results):

    comparison_table = Table(title=title)
    comparison_table.add_column("Model", justify="center", style="green", width=15)
    comparison_table.add_column("Accuracy", justify="right", style="green")
    comparison_table.add_column("Precision", justify="right", style="green")
    comparison_table.add_column("Recall", justify="right", style="green")
    comparison_table.add_column("F1", justify="right", style="green")

    for model, metrics in results.items():
        comparison_table.add_row(
            model,
            f'{metrics["accuracy_score"]:.4f}',
            f'{metrics["precision_score"]:.4f}',
            f'{metrics["recall_score"]:.4f}',
            f'{metrics["f1_score"]:.4f}',
        )

    console.print(comparison_table)

# 5. Evaluation

We train and validate the model with different architectures to choose the best one.

In [ ]:
input_size = X_train_tensor.shape[1]
output_size = 3

results = {}

for name, hidden_sizes in architectures.items():

    model = WineQualityNN(input_size, hidden_sizes, output_size)

    trained_model, train_loss, val_loss, val_acc = train_nn_model(
        model,
        X_train_tensor,
        y_train_tensor,
        X_val_tensor,
        y_val_tensor,
        epochs=200,
        lr=0.0005,
    )

    y_val_pred = evaluate_nn_model(trained_model, X_val_tensor)
    val_metrics = calculate_metrics(y_val_tensor.cpu().numpy(), y_val_pred)

    results[name] = val_metrics

In [ ]:
create_comparison_table(
    "Comparison of metrics between different architectures", results
)

Based on the data obtained, we will choose the best architecture according to its accuracy_score.

In [ ]:
best_model_name = max(results, key=lambda x: results[x]["accuracy_score"])
console.print(f"Best architecture: {best_model_name}")

We train the model with both training and validation data.

In [ ]:
X_train_full = torch.cat((X_train_tensor, X_val_tensor), dim=0)
y_train_full = torch.cat((y_train_tensor, y_val_tensor), dim=0)

best_architecture = architectures[best_model_name]

final_model = WineQualityNN(input_size, best_architecture, output_size)

final_model, _, _, _ = train_nn_model(
    final_model,
    X_train_full,
    y_train_full,
    X_train_full,
    y_train_full,
    epochs=100,
    lr=0.001,
)

y_test_pred = evaluate_nn_model(final_model, X_test_tensor)
test_metrics = {}
test_metrics[best_model_name] = calculate_metrics(
    y_test_tensor.cpu().numpy(), y_test_pred
)

In [ ]:
create_comparison_table("Best Architecture Metrics", test_metrics)

# 6. Analysis and Outlook

* **Which characteristics seem most important for wine quality classification?**
  Thanks to the correlation matrix, I was able to identify both positively correlated characteristics, such as sulfates and alcohol, which tend to increase with better quality, and negatively correlated characteristics, such as volatile acidity, which tend to decrease as wine quality improves.

* **How well does your model distinguish between the three quality classes?**
  According to the metrics obtained with the best architecture, I achieved an accuracy of 0.7267, which I consider an acceptable result. However, I believe that this performance could be improved with a larger amount of data and a better neural network configuration, which is an issue I still struggle with.

* **What challenges did you encounter?**
  One of the biggest challenges was defining the neural network architecture: the number of neurons, the activation functions, and the choice of hyperparameters. I found this part quite complex, so I largely relied on previous examples we had done with the data science professor to give me a reference for how to structure the model.

* **Which architectural choices or hyperparameters had the greatest impact on performance?**
  This was definitely the most decisive part. Small adjustments in the number of layers, learning rates, or activation functions can significantly change the performance of the model. Finding the right balance is key for the neural network to correctly identify patterns in the data.

* **Suggestions for improvement.**
  Although this topic was presented as a review, in practice it was not so simple. In the Machine Learning course, we looked at neural networks in a very general way, and it coincided with the capstone assignment, so we couldn't go into enough depth. I would like future classes to spend more time on understanding how to design the architecture of a neural network from scratch, as that continues to be the most challenging part for me.